In [1]:
from src.dm import DataModule

dm = DataModule(batch_size=4, num_workers=0, pin_memory=False)

dm.setup()

In [2]:
dm.test_df

,image,AOD
0,data/test_images/test_1.tif,1.442470
1,data/test_images/test_2.tif,0.910374
2,data/test_images/test_3.tif,0.500745
3,data/test_images/test_4.tif,0.999882
4,data/test_images/test_5.tif,1.668206
...,...,...
1484,data/test_images/test_1485.tif,1.597977
1485,data/test_images/test_1486.tif,1.819262
1486,data/test_images/test_1487.tif,1.334963
1487,data/test_images/test_1488.tif,0.231196


In [3]:
import os

os.listdir('checkpoints')

['cv_mlp-1-val_metric=0.96757-epoch=79.ckpt',
 'cv_mlp-2-val_metric=0.97834-epoch=389.ckpt',
 'cv_mlp-3-val_metric=0.90623-epoch=160.ckpt',
 'lrsch-val_metric=0.94095-epoch=461.ckpt',
 'cv_mlp-3-epoch=499.ckpt',
 'cv_mlp-2-epoch=499.ckpt',
 'cv_mlp-0-val_metric=0.97128-epoch=230.ckpt',
 'cv_mlp-0-epoch=499.ckpt',
 'cv_mlp-4-val_metric=0.92413-epoch=84.ckpt',
 'da_mlp-0-val_metric=0.97328-epoch=171.ckpt',
 'cv_mlp-1-epoch=499.ckpt',
 'lrsch-val_metric=0.94319-epoch=150.ckpt',
 'da-val_metric=0.94160-epoch=364.ckpt',
 'cv_mlp-4-epoch=499.ckpt',
 'cv1']

In [4]:
import torch 
from src.module import Module

names = [
	"cv_mlp-0-val_metric=0.97128-epoch=230.ckpt",
	"cv_mlp-1-val_metric=0.96757-epoch=79.ckpt",
	"cv_mlp-2-val_metric=0.97834-epoch=389.ckpt",
	"cv_mlp-3-val_metric=0.90623-epoch=160.ckpt",
	"cv_mlp-4-val_metric=0.92413-epoch=84.ckpt"
	
]

/home/juan/miniconda3/envs/peo/lib/python3.8/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/home/juan/miniconda3/envs/peo/lib/python3.8/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedba

In [5]:
class Identity:
    def __call__(self, x):
        return x

class Rot90:
    def __init__(self, axes=(2, 3)):
        self.axes = axes
    def __call__(self, x):
        return torch.rot90(x, 1, self.axes)

class Rot180:
    def __init__(self, axes=(2, 3)):
        self.axes = axes
    def __call__(self, x):
        return torch.rot90(x, 2, self.axes)

class Rot270:
    def __init__(self, axes=(2, 3)):
        self.axes = axes
    def __call__(self, x):
        return torch.rot90(x, 3, self.axes)

class Flip:
    def __init__(self, axis=2):
        self.axis = axis
    def __call__(self, x):
        return torch.flip(x, [self.axis])

class Transpose:
    def __init__(self, axes=(2, 3)):
        self.axes = axes
    def __call__(self, x):
        return torch.transpose(x, self.axes[0], self.axes[1])

trans = [
    (Identity(), Identity()),
    (Rot90(), Rot270()),
    (Rot180(), Rot180()),
    (Rot270(), Rot90()),
    (Flip(2), Flip(2)),
    (Flip(3), Flip(3)),
    (Transpose(), Transpose()),
]

In [6]:
from tqdm import tqdm
import numpy as np

trans = [
    Identity(),
    Rot90(), 
    Rot180(), 
    Rot270(), 
    Flip(2), 
    Flip(3), 
    Transpose()
]

device = "cuda:1"

cv_preds = []
for name in names:
    print(name)
    checkpoint = f'./checkpoints/{name}'
    checkpoint = torch.load(checkpoint, map_location='cpu')
    state_dict = checkpoint['state_dict']
    module = Module({'head_layers': [512, 256, 128, 1], 'freeze': True, 'p_drop': 0})
    module.load_state_dict(state_dict)
    module.to(device)
    module.eval()
    preds = []
    for batch in tqdm(dm.test_dataloader()):
        dict_of_tensors, _ = batch
        dict_of_tensors = {k: v.to(device) for k, v in dict_of_tensors.items()}
        x0 = dict_of_tensors['pixels'].clone()
        _preds = []
        with torch.no_grad():
            for t in trans:
                dict_of_tensors['pixels'] = t(x0)
                _preds.append(module(dict_of_tensors))
        output = torch.stack(_preds).mean(0).cpu()
        output = output * dm.aod_stats[1] + dm.aod_stats[0] 
        preds += output.cpu().tolist()
    cv_preds.append(preds)
preds = np.stack(cv_preds).mean(0)

cv_mlp-0-val_metric=0.97128-epoch=230.ckpt


100%|██████████| 373/373 [00:48<00:00,  7.65it/s]


cv_mlp-1-val_metric=0.96757-epoch=79.ckpt


100%|██████████| 373/373 [00:51<00:00,  7.28it/s]


cv_mlp-2-val_metric=0.97834-epoch=389.ckpt


100%|██████████| 373/373 [00:51<00:00,  7.28it/s]


cv_mlp-3-val_metric=0.90623-epoch=160.ckpt


100%|██████████| 373/373 [00:50<00:00,  7.37it/s]


cv_mlp-4-val_metric=0.92413-epoch=84.ckpt


100%|██████████| 373/373 [00:50<00:00,  7.38it/s]


In [7]:
dm.test_df['AOD'] = preds
dm.test_df.image = dm.test_df.image.apply(lambda x: x.split('/')[-1])

dm.test_df

,image,AOD
0,test_1.tif,0.082108
1,test_2.tif,0.210593
2,test_3.tif,0.213986
3,test_4.tif,0.112634
4,test_5.tif,0.077824
...,...,...
1484,test_1485.tif,0.092171
1485,test_1486.tif,0.139902
1486,test_1487.tif,0.108351
1487,test_1488.tif,0.073503


In [8]:
dm.test_df.to_csv('submission.csv', index=False, header=False)